In [26]:
import re
import io
import time
import json
import obonet
import fastobo
import sqlite3
import ahocorasick
import pandas as pd
from os import listdir
from os.path import isfile, join
from nltk.corpus import words

In [ ]:
# path = "./Papers"
# files = [f for f in listdir(path) if isfile(join(path, f))]

In [ ]:
# dfs = [] 

# for file in files:
#     sub_df = pd.read_excel(f"./Papers/{file}", skiprows=[0], index=False)
#     dfs.append(sub_df)

# df = pd.concat(dfs, ignore_index=True)
# df.reset_index(inplace=True)

In [ ]:
# copy_df = df.copy(deep=True)
# copy_df.to_excel("Papers.xlsx", index=False)

In [ ]:
# df.shape

In [ ]:
# [*df.columns]

In [ ]:
# # Drop Duplicates
# df.drop_duplicates(subset=["Title", "Abstract"], inplace=True, ignore_index=True)
# for column in ["DOI", "PMID", "PMCID", "ISBN"]:
#     df = df[(~df[column].duplicated()) | (df[column].isnull())]
# df.reset_index(inplace=True, drop=True)

In [ ]:
# df.shape

In [ ]:
# # Drop Rows w/ Null Title or Null Abstract
# df.dropna(subset=["Title", "Abstract"], inplace=True, ignore_index=True)
# df.reset_index(inplace=True, drop=True)

In [ ]:
# df.shape

In [ ]:
# # Store Papers' Fields of Research
# fields_of_research = df["Fields of Research (ANZSRC 2020)"].tolist()

# field_counts = {}
# fields_of_research_flat = []
# for field_of_research in fields_of_research:
#     fields = re.split(r";\s?", field_of_research)

#     # Update Count
#     for field in fields:
#         field_counts[field] = field_counts.get(field, 0) + 1
    
#     fields_of_research_flat.extend(fields)

In [ ]:
# field_counts_items = [*field_counts.items()]
# field_counts_items.sort(key=lambda field_count: field_count[1], reverse=True)
# fields_of_research_flat_set = set(fields_of_research_flat)

In [ ]:
# bad_fields_of_research = [
#     "3101 Biochemistry and Cell Biology", 
#     "3105 Genetics", 
#     "34 Chemical Sciences", 
#     "3401 Analytical Chemistry", 
#     "3215 Reproductive Medicine",
#     "3215 Reproductive Medicine",
#     "33 Built Environment and Design",
#     "34 Chemical Sciences",
#     "3401 Analytical Chemistry",
#     "3402 Inorganic Chemistry",
#     "3403 Macromolecular and Materials Chemistry",
#     "3404 Medicinal and Biomolecular Chemistry",
#     "3406 Physical Chemistry",
#     "40 Engineering",
#     "4001 Aerospace Engineering",
#     "4003 Biomedical Engineering",
#     "4004 Chemical Engineering",
#     "4016 Materials Engineering",
#     "3402 Inorganic Chemistry",
#     "3703 Geochemistry",
#     "32 Biomedical and Clinical Sciences",
#     "3207 Medical Microbiology",
#     "3106 Industrial Biotechnology",
# ]

In [ ]:
# Peek at Abstracts
# sub_df = df[df["Fields of Research (ANZSRC 2020)"].str.contains('32 Biomedical and Clinical Sciences')]
# sub_df.reset_index(inplace=True, drop=True)

# for i in range(min(10, len(sub_df))):
#     print(sub_df.loc[i, "Title"])
#     print(sub_df.loc[i, "Abstract"])
#     print()
#     print()

In [ ]:
# max_sum = 0
# for field, count in field_counts_items:
#     print(f"{field}: {count}")
#     if field in bad_fields_of_research:
#         max_sum += count
# print(f"Maximum Removed: {max_sum}")

In [ ]:
# df_without_bad_fields_of_research = df[~df["Fields of Research (ANZSRC 2020)"].str.contains(rf"{'|'.join(bad_fields_of_research)}")]
# df_without_bad_fields_of_research.shape

In [ ]:
# def load_obo(path):
#     try:
#         return fastobo.load(path)
#     except SyntaxError:
#         print(f"fastobo failed for {path}, falling back to manual parse")

In [ ]:
c_doc = fastobo.load("./Ontology/chebi_core.obo")
p_doc = fastobo.load("./Ontology/pro_reasoned.obo")
g_doc = fastobo.load("./Ontology/go-basic.obo")
# t_doc = fastobo.load("./Ontology/to.obo") # if obsolete in name, don't use, kind of like traits but chemical so nvm
# s_doc = obonet.read_obo("./Ontology/so.obo") # doesnt really have names so not gonna use

In [45]:
conn = sqlite3.connect("COL.db")
cursor = conn.cursor()

In [48]:
def is_species(name):
    query = f"""
        SELECT NameUsage."col:scientificName"
        FROM NameUsage
        WHERE 
            LOWER(NameUsage."col:scientificName") = LOWER('{name}')
        LIMIT 1
    """
    output = cursor.execute(query).fetchone()
    return output

def re_is_scientific_name(name: str, flags: int = 0) -> bool:
    return re.match(r"^[A-Z][a-z]+\s[a-z]+$", name, flags)

def re_is_scientific_name_abbrev(name: str, flags: int = 0):
    if not re.match(r"^[A-Z]\.\s[a-z]+$", name, flags):
        return False
    return name.split()

In [51]:
bool(is_species("Panthera ldsseo"))

False

In [92]:
names = []
for frame in p_doc:
    if not isinstance(frame, fastobo.term.TermFrame):
        continue
        
    name = None
    
    for clause in frame:
        if isinstance(clause, fastobo.term.NameClause):
            name = str(clause.name)
            names.append(name)
            break

print("Number Names: ", len(names))

Number Names:  365852


In [95]:
do_not_use_names = []

for name in names:
    if 'thaliana' in name and '(' not in name:
        print(name)
        
    # if (re_is_scientific_name(name) and is_species(name)):
    #     do_not_use_names.append(name)
    #     do_not_use_names.extend(name.split())
    #     print(name)

Arabidopsis thaliana
Arabidopsis thaliana protein


In [96]:
is_species('Arabidopsis thaliana')

('Arabidopsis thaliana',)

In [ ]:
sub_names = []
for name in names:
    if len(name) <= 2:
        sub_names.append(name)

In [ ]:
len(sub_names)

In [ ]:
sub_names[:1000]

In [98]:
eng_words = set(w.lower() for w in words.words())

In [99]:
'thaliana' in eng_words

False

# 'cell' in eng_words

In [ ]:
from wordfreq import word_frequency
word_frequency("taxa", "en")

In [ ]:
print(f"{float(2.45e-05):2f}")

In [ ]:
print(f"{float(1e-06):2f}")

In [ ]:
A = ahocorasick.Automaton()

for doc in [c_doc, p_doc]:
    for frame in doc:
        if not isinstance(frame, fastobo.term.TermFrame):
            continue
            
        name = None
        
        for clause in frame:
            if isinstance(clause, fastobo.term.NameClause):
                name = str(clause.name)
                break

        if name == 'sulfur':
            print(frame)
            
        if not name:
            continue
        
        if len(name) <= 2 or len(name) >= 20:
            continue
    
        name_mod = re.sub(r"(\(|\[).*(\)|\])$", "", name).strip()
        name_mod = re.sub(r"^-", "", name_mod)
    
        if any([bool(word in eng_words) for word in name.lower().split()]):
            continue

        if len(name) <= 2:
            continue

        if word_frequency(name, "en") < 1e-6:
            A.add_word(name, name)

        if any([bool(word in eng_words) for word in name_mod.lower().split()]):
            continue
        
        if len(name_mod) <= 2:
            continue

        if word_frequency(name_mod, "en") < 1e-6:
            A.add_word(name_mod, name_mod)

A.make_automaton()

In [ ]:
# chebi chemical very very very long terms, maybe extract words
# pro_reasoned need to remove everything in brackets

In [ ]:
texts = df_without_bad_fields_of_research.Abstract.tolist()

mask = []
names = set()

for text in texts:
    found = False

    term_id = None
    for end_index, term_id in A.iter(text):
        start_index = end_index - len(term_id) + 1

        before = start_index == 0 or not text[start_index-1].isalpha()
        after = end_index + 1 == len(text) or not text[end_index+1].isalpha()

        if before and after:
            found = True
            print(f"Found '{term_id}' at position {start_index}-{end_index}")
            break
    
    if found:
        mask.append(False)

        if term_id not in names:
            print(text)
            print()

        names.add(term_id)
        
        # if len(names) >= 50:
        #     break
    else:
        mask.append(True)    

In [ ]:
df_without_bad_abstracts = df_without_bad_fields_of_research[mask]

In [101]:
from wordfreq import word_frequency

test_words = ["lipid", "thaliana", "pigs", "viruses", "flavonoid", "Caenorhabditis", "hid", "sulfur", "carbon", "nitrogen", "water", "iron", "glucose", "medicarpin"]
for w in test_words:
    print(w, word_frequency(w, "en"))
    if word_frequency(w, "en") < 1e-6:
        print("\treject")
    else:
        print("\taccept")

lipid 1.95e-06
	accept
thaliana 1.41e-07
	reject
pigs 9.77e-06
	accept
viruses 5.37e-06
	accept
flavonoid 1.05e-07
	reject
Caenorhabditis 1.07e-07
	reject
hid 6.17e-06
	accept
sulfur 2.69e-06
	accept
carbon 3.55e-05
	accept
nitrogen 5.75e-06
	accept
water 0.000331
	accept
iron 5.5e-05
	accept
glucose 5.62e-06
	accept
medicarpin 0.0
	reject


In [103]:
'arabidopsis' in eng_words

True

In [ ]:
df_without_bad_abstracts.shape

In [ ]:
df_with_bad_abstracts = df_without_bad_fields_of_research[[not m for m in mask]]

In [ ]:
df_with_bad_abstracts.shape

In [ ]:
df_with_bad_abstracts.iloc[0].Abstract

In [ ]:
df = pd.read_excel("../Dimensions-Publication-2026-01-22_18-49-40.xlsx", skiprows=[0])
print(f"Shape: {df.shape}")

In [ ]:
!python -m pip install english-words

In [59]:
from english_words import get_english_words_set

In [60]:
web2lowerset = get_english_words_set(['gcide', 'web2'], lower=True)

In [104]:
"arabidopsis" in web2lowerset

True

In [63]:
!python -m pip install PyDictionary

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for goslate: filename=goslate-1.5.4-py3-none-any.whl size=11673 sha256=b7f03ec6ce0998d444197d8e29cad666b850cb5ee33bd37102c7857a0a8cf2ce
  Stored in directory: c:\users\lbeln\appdata\local\pip\cache\wheels\b5\30\e9\63b6de83667be2977ee793a146a2c80f8e588d5c0203b39dc9
  Created wheel for futures: filename=futures-3.0.5-py3-non

    extract-msg (<=0.29.*)
                 ~~~~~~~^

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [64]:
from PyDictionary import PyDictionary

In [66]:
dictionary = PyDictionary()

In [74]:
dictionary.meaning("pet")

{}

In [75]:
print(dictionary.meaning("indentation"))

{}


In [76]:
!python -m pip install pymultidictionary


   ---------------------------------------- 2/2 [pymultidictionary]



    extract-msg (<=0.29.*)
                 ~~~~~~~^

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [78]:
from PyMultiDictionary import MultiDictionary
dictionary = MultiDictionary()
print(dictionary.meaning('en', 'good'))

HTTPError: HTTP Error 403: Forbidden

In [83]:
!python -m pip install pyenchant

   ---------------------------------------- 0.0/37.4 MB ? eta -:--:--
    --------------------------------------- 0.5/37.4 MB 8.5 MB/s eta 0:00:05
   ------- -------------------------------- 7.3/37.4 MB 28.3 MB/s eta 0:00:02
   -------------- ------------------------- 13.4/37.4 MB 28.9 MB/s eta 0:00:01
   ----------------------- ---------------- 22.3/37.4 MB 32.8 MB/s eta 0:00:01
   -------------------------------- ------- 30.7/37.4 MB 34.2 MB/s eta 0:00:01
   ---------------------------------------  37.2/37.4 MB 35.3 MB/s eta 0:00:01
   ---------------------------------------- 37.4/37.4 MB 33.0 MB/s  0:00:01


    extract-msg (<=0.29.*)
                 ~~~~~~~^

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [85]:
import enchant

d = enchant.Dict("en_US")

# Check if a word is valid
print(d.check("pet"))       # True
print(d.check("ptee"))      # False

# Get spelling suggestions
print(d.suggest("ptee"))    # ['tree', 'pee', 'tee', ...]

AttributeError: module 'enchant' has no attribute 'Dict'